In [1]:
%cd ../../

/home/hoanghu/projects/fw-models


In [2]:
from pathlib import Path

import joblib
import numpy as np
import polars as pl
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
from sklearn.preprocessing import TargetEncoder, OrdinalEncoder
# from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
# from xgboost import XGBRegressor
# from catboost import CatBoostRegressor
# from lightgbm import LGBMRegressor
# from sklearn.metrics import root_mean_squared_error, r2_score
import lightning as L
import torch.nn.functional as F
from torch import nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import torch
from torch.nn import Module
from torch import Tensor
from torchmetrics.regression import MeanSquaredError, R2Score
from lightning.pytorch.callbacks import RichProgressBar
from lightning.pytorch.loggers import TensorBoardLogger
from sklearn.preprocessing import MinMaxScaler
from polars import DataFrame

In [3]:
# plt.style.use('seaborn-v0_8')
# plt.rcParams.update({'font.size': 8})

# Load data and resources

In [4]:
path = "data/processed/phase_4/dim_pieces_per_dish.xlsx"

pos = pl.read_excel(path)
pos.head()

date,restaurant,meal_type,pcs,meal_id
date,str,str,i64,i64
2023-01-02,"""che""","""fish""",78,9500047
2023-01-02,"""che""","""vegan""",84,6128
2023-01-02,"""che""","""meat""",165,9500139
2023-01-03,"""che""","""vegetarian""",29,1270
2023-01-03,"""che""","""fish""",105,6156


In [5]:
path = "notebooks/phase_4/2_foreacast_pieces_per_dish/data/Nov22_meals_cosine_sim.parquet"
cosine_sim = pl.read_parquet(path)

cosine_sim.head()

meal_id_x,meal_id_y,cosine_sim,__index_level_0__
i64,i64,f64,i64
34,37,0.040771,1
34,710,0.072106,2
34,713,0.031979,3
34,724,0.063252,4
34,725,0.026991,5


In [6]:
path = "data/processed/phase_4/dim_meals.xlsx"

dim_meals = pl.read_excel(path)
dim_meals.head()

meal_id,meal_type_1,schoolyear,is_kela,is_new,restaurant,meal_type_2,pcs_mean
i64,str,str,bool,bool,str,str,f64
9017,"""vegan""","""24-25""",true,true,"""che-exa""","""vegan-miscellaneous""",140.165854
7201,"""vegan""","""23-24""",false,false,null,null,121.918212
9032,"""vegan""","""23-24""",false,false,null,null,121.918212
9102,"""vegan""","""23-24""",false,false,null,null,121.918212
7010,"""vegetarian""","""24-25""",false,false,"""che-exa""",null,71.0


# Process dim tables

In [7]:
# Filter out too low POS values
THETA = 5
pos = pos.filter(pl.col('pcs') >= THETA)

# Keep THETA most consumed meals per date-restaurant
THETA = 5
pos = (
    pos
    .with_columns(pl.col('pcs').rank(descending=True, method='dense').over("date", 'restaurant').alias('rank'))
    .filter(pl.col("rank") <= THETA)
    .drop('rank')
)


pos.head()

date,restaurant,meal_type,pcs,meal_id
date,str,str,i64,i64
2023-01-02,"""che""","""fish""",78,9500047
2023-01-02,"""che""","""vegan""",84,6128
2023-01-02,"""che""","""meat""",165,9500139
2023-01-03,"""che""","""vegetarian""",29,1270
2023-01-03,"""che""","""fish""",105,6156


## Build encoders

In [8]:
CUTOFF_DATE = "2024-10-01"
pos_train = pos.filter(pl.col('date') < pl.lit(CUTOFF_DATE).str.to_datetime())

enc_meal_id = TargetEncoder(target_type='continuous')
enc_meal_id.fit(pos_train.select('meal_id'), pos_train.get_column('pcs'))

enc_restaurant = OrdinalEncoder()
enc_restaurant.fit(pos.select('restaurant'))

enc_meal_type = OrdinalEncoder()
enc_meal_type.fit(pos.select('meal_type'))

OrdinalEncoder()

# Build feature

## For each meal, keep `K` closest meals and the closest meals must appear in the POS data

In [9]:
K = 5

In [10]:
meals_type = dim_meals.select('meal_id', pl.col('meal_type_1').alias('meal_type'))
meal_ids_valid = pos.get_column('meal_id').unique()

In [11]:
meals_similar = (
    cosine_sim

    # Only consider meals appearing in POS data
    .filter(
        (pl.col('meal_id_x').is_in(meal_ids_valid))
        & (pl.col('meal_id_y').is_in(meal_ids_valid))
    )

    
    .join(meals_type, left_on='meal_id_x', right_on='meal_id', how='left')
    .rename({'meal_type': 'meal_type_x'})
    .join(meals_type, left_on='meal_id_y', right_on='meal_id', how='left')
    .filter(pl.col('meal_type_x') == pl.col('meal_type'))

    .with_columns(
        pl.col('cosine_sim').rank('dense', descending=True).over('meal_id_x').alias('rank')
    )
    .filter(pl.col("rank") <= K)
    .select('meal_id_x', 'meal_id_y', 'cosine_sim')
)


# Add entries for each meal with itself
ids = meals_similar.select(pl.col('meal_id_x').unique()).get_column('meal_id_x')
tmp = pl.DataFrame({
    'meal_id_x': ids,
    'meal_id_y': ids,
    'cosine_sim': 1.0
})
meals_similar_all = pl.concat([meals_similar, tmp])



# Sample features

In [12]:
base_meal_similar = (
    meals_similar_all
    .group_by('meal_id_x')
    .agg(
        pl.concat_list('meal_id_y').flatten()
    )
)

base_meals = (
    pos
    .group_by('date', 'restaurant')
    .agg(
        pl.concat_list('meal_id').flatten()
    )
)

In [13]:
list_df = []

for i in range(100):
    # Sample similar meals
    sample_meals_similar = (
        base_meal_similar
        .select(
            'meal_id_x',
            pl.col('meal_id_y').list.sample(1).flatten()
        )
        .join(meals_similar_all, on=['meal_id_x', 'meal_id_y'], how='left')
        .rename({'meal_id_x': 'meal_id_primary', 'meal_id_y': 'meal_id_sub'})
    )

    # Sample POS
    sample_pos = (
        base_meals
        .select(
            'date', 'restaurant',
            pl.col('meal_id')
                .list
                .sample(fraction=1., with_replacement=False, shuffle=True)
                    .alias('meal_id_primary')
        )
        .with_row_index()
        .explode('meal_id_primary')
    )

    # Join
    sample_pos = (
        sample_pos
        .join(sample_meals_similar, on='meal_id_primary', how='left')
        .select(
            pl.concat_str((pl.lit(f"{i}-"), pl.col('index').cast(pl.String))).alias('index'),
            'date',
            'restaurant',
            'meal_id_primary',
            pl.col('meal_id_sub').alias('meal_id'),
            pl.col('cosine_sim').alias('sim')
        )
    )

    list_df.append(sample_pos)

pos_final = pl.concat(list_df)



# Supplement some info
pos_final = (
    pos_final
    .join(pos.rename({'meal_id': 'meal_id_primary'}), on=['date', 'restaurant', 'meal_id_primary'])
    .drop('meal_id_primary')
)



pos_final.head()

index,date,restaurant,meal_id,sim,meal_type,pcs
str,date,str,i64,f64,str,i64
"""0-0""",2024-09-10,"""exa""",7013,0.120195,"""vegetarian""",55
"""0-0""",2024-09-10,"""exa""",6096,0.234124,"""vegan""",169
"""0-0""",2024-09-10,"""exa""",6636,0.131375,"""fish""",143
"""0-0""",2024-09-10,"""exa""",9500063,0.230166,"""vegan""",74
"""0-1""",2024-10-08,"""che""",9500089,0.074267,"""fish""",379


# Train

## Use ML models

### Apply encoding

In [14]:
# pos_final = (
#     pos_final
#     .with_columns(
#         pl.col('date').dt.weekday().alias('weekday'),
#         pl.col('date').dt.day().alias('day'),
#         pl.col('date').dt.month().alias('month'),
#     )
#     .with_columns(
#         (pl.col('weekday') * 2 * np.pi / 7).sin().alias('weekday_sin'),
#         (pl.col('weekday') * 2 * np.pi / 7).cos().alias('weekday_cos'),
#         (pl.col('day') * 2 * np.pi / 31).sin().alias('day_sin'),
#         (pl.col('day') * 2 * np.pi / 31).cos().alias('day_cos'),
#         (pl.col('month') * 2 * np.pi / 12).sin().alias('month_sin'),
#         (pl.col('month') * 2 * np.pi / 12).cos().alias('month_cos'),
#     )
#     .drop('weekday', 'day', 'month')
# )


# pos_final = (
#     pos_final
#     .with_columns(
#         pl.Series('meal_id_enc', values=enc_meal_id.transform(pos_final.select('meal_id'))).arr.first(),
#         # pl.col('meal_id').alias('meal_id_enc').cast(pl.Int32),
#         pl.Series('meal_type_enc', values=enc_meal_type.transform(pos_final.select('meal_type'))).arr.first().cast(pl.Int32),
#         pl.Series('restaurant_enc', values=enc_restaurant.transform(pos_final.select('restaurant'))).arr.first().cast(pl.Int32),
#     )
# )


# pos_final.head()

In [15]:
# cols_X = [
#     'restaurant_enc',
#     'meal_type_enc',
#     'meal_id_enc',
#     'sim',
#     'weekday_sin',
#     'weekday_cos',
#     'day_sin',
#     'day_cos',
#     'month_sin',
#     'month_cos'
# ]

# cols_y = 'pcs'
# cols_ypred = 'pcs_pred'


# cols_cat = [
#     'meal_id_enc',
#     'meal_type_enc',
#     'restaurant_enc',
# ]


# train = pos_final.filter(pl.col('date') < pl.lit(CUTOFF_DATE).str.to_datetime()).to_pandas()
# val = pos_final.filter(pl.col('date') >= pl.lit(CUTOFF_DATE).str.to_datetime()).to_pandas()

# Xtrain, ytrain = train[cols_X] , train[cols_y]
# Xval, yval = val[cols_X], val[cols_y]

In [16]:
# models = {
#     # 'RF': {'model': RandomForestRegressor(), 'is_cat': False},
#     # 'GB': {'model': GradientBoostingRegressor(), 'is_cat': False},
#     'XGB':      {'model': XGBRegressor(), 'is_cat': False},
#     'XGB-cat':  {'model': XGBRegressor(tree_method="hist", enable_categorical=True), 'is_cat': True},
#     'CatBoost': {'model': CatBoostRegressor(verbose=False, cat_features=cols_cat), 'is_cat': True},
#     'LBGM':     {'model': LGBMRegressor(verbose=0), 'is_cat': False},
#     'LBGM-cat': {'model': LGBMRegressor(verbose=0), 'is_cat': True},
# }

# for name, model in models.items():
#     print(f"Start: {name}")

#     X = Xtrain.copy()

#     if model['is_cat'] is True:
#         for col in cols_cat:
#             X[col] = X[col].astype('category')

#     if name == 'LBGM-cat':
#         kwargs = dict(categorical_feature=cols_cat)
#     else:
#         kwargs = {}
#     model['model'].fit(X, ytrain, **kwargs)

In [17]:
# metrics_models = []

# def calc_metrics(split: pd.DataFrame, model: dict, verbose: bool = False) -> tuple:
#     X = split[cols_X].copy()
#     if model['is_cat'] is True:
#         for col in cols_cat:
#             X[col] = X[col].astype('category')

#     split['pcs_pred'] = np.clip(model['model'].predict(X), a_min=0, a_max=None)
#     split = (
#         split
#         .groupby(['date', 'restaurant', 'meal_id', 'meal_type'])
#         .agg({'pcs': 'mean', 'pcs_pred': 'mean'})
#         .reset_index()
#     )
    
    
#     rmse = root_mean_squared_error(split['pcs'], split['pcs_pred']).item()
#     r2 = r2_score(split['pcs'], split['pcs_pred'])

#     if verbose is True:
#         print(f"rmse: {rmse:.4f}")
#         print(f"r2  : {r2:.4f}")

#     return rmse, r2

# print("== Split: val")
# for name, model in models.items():
#     print(f"Start: {name}")

#     rmse, r2 = calc_metrics(val, model)
#     metrics_models.append({'model': name, 'val': rmse, 'split': 'val', 'metric': 'rmse'})
#     metrics_models.append({'model': name, 'val': r2, 'split': 'val', 'metric': 'r2'})


# print("== Split: train")
# for name, model in models.items():
#     print(f"Start: {name}")

#     rmse, r2 = calc_metrics(train, model)
#     metrics_models.append({'model': name, 'val': rmse, 'split': 'train', 'metric': 'rmse'})
#     metrics_models.append({'model': name, 'val': r2, 'split': 'train', 'metric': 'r2'})



# # Visualize
# metrics = pd.DataFrame.from_records(metrics_models)

# fig = plt.figure(figsize=(12, 8))
# fig.suptitle("Metrics", fontsize=15, fontweight='bold')
# fig.subplots_adjust(wspace=0.2, hspace=0.3)

# for i, (split, metric) in enumerate(product(['train', 'val'], ['rmse', 'r2'])):
#     df = metrics[(metrics['split'] == split) & (metrics['metric'] == metric)]

#     ax = fig.add_subplot(2, 2, i+1)
#     sns.barplot(df, x='model', y='val', ax=ax)
#     description = "lower" if metric == "rmse" else "higher"
#     ax.set_title(f"{split} - {metric} ({description} is better)")
#     ax.set_xlabel("")


## Use DL

In [18]:
pos_final.head()

index,date,restaurant,meal_id,sim,meal_type,pcs
str,date,str,i64,f64,str,i64
"""0-0""",2024-09-10,"""exa""",7013,0.120195,"""vegetarian""",55
"""0-0""",2024-09-10,"""exa""",6096,0.234124,"""vegan""",169
"""0-0""",2024-09-10,"""exa""",6636,0.131375,"""fish""",143
"""0-0""",2024-09-10,"""exa""",9500063,0.230166,"""vegan""",74
"""0-1""",2024-10-08,"""che""",9500089,0.074267,"""fish""",379


### Apply encoder and scaler

In [ ]:
scaler = MinMaxScaler((0.1, 1.0))
scaler.fit(pos_train.select('pcs'))

In [20]:
# Encode fields
pos_final = (
    pos_final
    .with_columns(
        pl.Series('meal_id_enc', values=enc_meal_id.transform(pos_final.select('meal_id'))).arr.first(),
        # pl.col('meal_id').alias('meal_id_enc').cast(pl.Int32),
        pl.Series('meal_type_enc', values=enc_meal_type.transform(pos_final.select('meal_type'))).arr.first().cast(pl.Int32),
        pl.Series('restaurant_enc', values=enc_restaurant.transform(pos_final.select('restaurant'))).arr.first().cast(pl.Int32),
    )
)
    
# Scale fields related to pcs
pos_final = (
    pos_final
    .with_columns(
        pl.Series('meal_id_enc', values=scaler.transform(pos_final.select(pl.col('meal_id_enc').alias('pcs')))).arr.first(),
        pl.Series('pcs_enc', values=scaler.transform(pos_final.select('pcs'))).arr.first(),
    )
)

# group-by
pos_final = (
    pos_final
    .group_by('index', 'date', 'restaurant_enc')
    .agg(
        pl.concat_list('meal_id_enc').flatten(),
        pl.concat_list('meal_type_enc').flatten(),
        pl.concat_list('sim').flatten(),
        pl.concat_list('pcs').flatten(),
        pl.concat_list('pcs_enc').flatten(),
    )
)


# Encode datetime
pos_final = (
    pos_final
    .with_columns(
        pl.col('date').dt.weekday().alias('weekday'),
        pl.col('date').dt.day().alias('day'),
        pl.col('date').dt.month().alias('month'),
    )
    .with_columns(
        (pl.col('weekday') * 2 * np.pi / 7).sin().alias('weekday_sin'),
        (pl.col('weekday') * 2 * np.pi / 7).cos().alias('weekday_cos'),
        (pl.col('day') * 2 * np.pi / 31).sin().alias('day_sin'),
        (pl.col('day') * 2 * np.pi / 31).cos().alias('day_cos'),
        (pl.col('month') * 2 * np.pi / 12).sin().alias('month_sin'),
        (pl.col('month') * 2 * np.pi / 12).cos().alias('month_cos'),
    )
    .drop('weekday', 'day', 'month')
)


pos_final.head()

index,date,restaurant_enc,meal_id_enc,meal_type_enc,sim,pcs,pcs_enc,weekday_sin,weekday_cos,day_sin,day_cos,month_sin,month_cos
str,date,i32,list[f64],list[i32],list[f64],list[i64],list[f64],f64,f64,f64,f64,f64,f64
"""90-909""",2024-01-26,1,"[0.273377, 0.268044, 0.106417]","[0, 3, 4]","[0.134536, 0.248848, 0.168538]","[132, 130, 43]","[0.31566, 0.312264, 0.164528]",-0.974928,-0.222521,-0.848644,0.528964,0.5,0.866025
"""45-690""",2023-03-03,0,"[0.310047, 0.346226, 0.362777]","[3, 2, 1]","[0.193394, 0.182169, 1.0]","[128, 199, 99]","[0.308868, 0.429434, 0.259623]",-0.974928,-0.222521,0.571268,0.820763,1.0,6.1232e-17
"""98-562""",2023-11-07,0,"[0.29007, 0.158337, … 0.106417]","[1, 3, … 4]","[1.0, 0.119013, … 0.168519]","[318, 223, … 10]","[0.631509, 0.470189, … 0.108491]",0.974928,-0.222521,0.988468,0.151428,-0.5,0.866025
"""17-195""",2023-09-14,1,"[0.11611, 0.123204, 0.310047]","[4, 0, 3]","[0.075151, 0.166679, 0.242877]","[15, 120, 144]","[0.116981, 0.295283, 0.336038]",-0.433884,-0.900969,0.299363,-0.954139,-1.0,-1.8370e-16
"""2-225""",2024-03-18,1,"[0.256814, 0.376792]","[0, 3]","[0.158702, 0.259364]","[115, 196]","[0.286792, 0.42434]",0.781831,0.62349,-0.485302,-0.874347,1.0,6.1232e-17


In [21]:
pos_train = pos_final.filter(pl.col('date') < pl.lit(CUTOFF_DATE).str.to_datetime())
pos_val = pos_final.filter(pl.col('date') >= pl.lit(CUTOFF_DATE).str.to_datetime())

In [22]:
pos_train.write_parquet("pos_train.parquet")
pos_val.write_parquet("pos_val.parquet")

### Prepare `Dataset` instance

In [3]:
PATH_SCALER = Path("res/scaler.save")
PATH_TRAIN = Path("data/inter/pos_train.parquet")
PATH_VAL = Path("data/inter/pos_val.parquet")

scaler = joblib.load(PATH_SCALER)

pos_train = pl.read_parquet(PATH_TRAIN)
pos_val = pl.read_parquet(PATH_VAL)

# pcs = (
#     pos_val
#     .select('date', 'restaurant_enc', 'pcs', 'pcs_enc')
#     .explode('pcs', 'pcs_enc')
# )

# pcs = (
#     pcs
#     .with_columns(
#         pl.Series('pcs_inv', values=scaler.inverse_transform(pcs.select('pcs_enc'))).arr.first()
#     )
# )

# pcs.head()

In [4]:
THETA = 5

class POSData(Dataset):
    def __init__(self, ds: DataFrame, device: str = 'cpu') -> None:
        super().__init__()

        self._ds = ds
        self._device = torch.device(device)

    def __getitem__(self, idx):
        record = self._ds.row(idx, named=True)

        restaurant = torch.tensor(record['restaurant_enc'], dtype=torch.int32)

        n = len(record['meal_id_enc'])
        n_zeros_padded = THETA - n

        meal = F.pad(torch.tensor(record['meal_id_enc'], dtype=torch.float32), (0, n_zeros_padded))
        meal_type = F.pad(torch.tensor(record['meal_type_enc'], dtype=torch.int32), (0, n_zeros_padded))
        sim = F.pad(torch.tensor(record['sim'], dtype=torch.float32), (0, n_zeros_padded))
        tgt_train = F.pad(torch.tensor(record['pcs_enc'], dtype=torch.float32), (0, n_zeros_padded))
        tgt = F.pad(torch.tensor(record['pcs'], dtype=torch.float32), (0, n_zeros_padded))

        # mask = torch.zeros((THETA, THETA), dtype=torch.float32)
        # mask[:n, :n] = 1.0
        mask = (meal == 0).clone().detach()

        date = torch.tensor(
            [record['weekday_sin'], record['weekday_cos'], record['day_sin'],
            record['day_cos'], record['month_sin'],record['month_cos'],],
            dtype=torch.float32
        )


        out = {
            'meal': meal,
            'meal_type': meal_type,
            'sim': sim,
            'mask': mask,
            'restaurant': restaurant,
            'date': date,
            'tgt_train': tgt_train,
            'tgt': tgt
        }

        return out

    def __len__(self,) -> int:
        return len(self._ds)

# train_ds = POSData(pos_train)
# # X = train_ds[0]
# loader_train = DataLoader(train_ds, batch_size=10, shuffle=True)
# for X in loader_train:
#     break

# X

### Prepare model

In [5]:
class POSForecast(Module):
    def __init__(
        self, n_meal_types: int = 5, n_restaurants: int = 3, d_hid: int = 32
    ) -> None:
        super().__init__()

        self._embd_meal_type = nn.Embedding(n_meal_types, d_hid)
        self._embd_restaurant = nn.Embedding(n_restaurants, d_hid)
        self.lin_date = nn.Linear(6, d_hid)
        self.lin_sim = nn.Linear(1, d_hid)
        self.lin_meal = nn.Linear(1, d_hid)

        self.ff_combine = nn.Sequential(
            nn.Linear(d_hid * 5, d_hid * 5),
            nn.Dropout(),
            nn.Tanh(),
            # nn.LayerNorm(d_hid * 5),
        )

        self.trans_encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=d_hid * 5, nhead=1, batch_first=True),
            num_layers=4,
            enable_nested_tensor=False,
        )

        self.lin_pcs = nn.Linear(d_hid * 5, 1)

    def forward(self, X: dict[str, Tensor]) -> Tensor:
        meal = X["meal"]
        meal_type = X["meal_type"]
        mask = X["mask"]
        restaurant = X["restaurant"]
        date = X["date"]
        sim = X["sim"]

        # Encode
        meal_type = self._embd_meal_type(meal_type)
        restaurant = self._embd_restaurant(restaurant)

        date = self.lin_date(date)
        sim = self.lin_sim(sim.unsqueeze(-1))
        meal = self.lin_meal(meal.unsqueeze(-1))

        # Concate fields
        N = meal.shape[1]
        restaurant = torch.repeat_interleave(restaurant.unsqueeze(1), N, dim=1)
        date = torch.repeat_interleave(date.unsqueeze(1), N, dim=1)

        meals = torch.concat(
            [
                meal,
                meal_type,
                restaurant,
                date,
                sim,
            ],
            dim=-1,
        )

        meals = self.ff_combine(meals)

        # Use Transformer Encoder
        SEQ_LEN = mask.shape[-1]
        mask = mask.unsqueeze(1).repeat_interleave(SEQ_LEN, dim=1)
        meals = self.trans_encoder(meals, mask)

        # meal_main = meals[:, 0:1]       # [bz, 1, d_hid * 4]
        # meals_other = meals[:, 1:]      # [bz, N-1, d_hid * 4]
        # S: Tensor = meal_main @ meals_other.permute(0, 2, 1)        # [bz, 1, N-1]
        # attentive_prob = nn.functional.softmax(S.masked_fill_(mask.unsqueeze(1), -1e10), dim=-1)
        # # [bz, 1, N-1]
        # h = attentive_prob @ meals_other
        # # [bz, 1, d_hid * 4]

        # meal_main = torch.concat([meal_main, h], dim=-1)
        # meal_main = self.lin2(meal_main)
        # [bz, 1, d_hid * 8]

        # Predict pos
        pos = self.lin_pcs(meals).squeeze(-1)
        pos = (~X['mask']).type(torch.float32) * pos
        
        return pos


train_ds = POSData(pos_train)
# X = train_ds[0]
loader_train = DataLoader(train_ds, batch_size=10, shuffle=True)
for X in loader_train:
    break

# model = POSForecast()

# out = model(X)
# out
# meal = X['meal']
# meal_type = X['meal_type']
# mask = X['mask']
# restaurant = X['restaurant']
# date = X['date']
# sim = X['sim']

# # Encode
# meal_type = model._embd_meal_type(meal_type)
# restaurant = model._embd_restaurant(restaurant)

# date = model.lin_date(date)
# sim = model.lin_sim(sim.unsqueeze(-1))
# meal = model.lin_meal(meal.unsqueeze(-1))

# # Concate fields
# N = meal.shape[1]
# restaurant = torch.repeat_interleave(restaurant.unsqueeze(1), N, dim=1)
# date = torch.repeat_interleave(date.unsqueeze(1), N, dim=1)

# meals = torch.concat(
#     [
#         meal,
#         meal_type,
#         restaurant,
#         date,
#         sim,
#     ],
#     dim=-1
# )

# meals = model.ff_combine(meals)

# # Use Transformer Encoder
# SEQ_LEN = mask.shape[-1]
# mask = mask.unsqueeze(1).repeat_interleave(SEQ_LEN, dim=1)


In [14]:
X['tgt_train']

tensor([[0.1068, 0.1119, 0.0000, 0.0000, 0.0000],
        [0.2562, 0.3309, 0.0000, 0.0000, 0.0000],
        [0.4685, 0.1951, 0.4481, 0.2342, 0.0000],
        [0.1102, 0.1068, 0.1272, 0.1153, 0.1170],
        [0.1323, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2902, 0.6994, 0.5126, 0.0000, 0.0000],
        [0.1221, 0.6587, 0.9694, 0.3377, 0.0000],
        [0.4957, 0.4294, 0.0000, 0.0000, 0.0000],
        [0.1068, 0.1272, 0.1102, 0.1170, 0.1153],
        [0.3394, 0.3649, 0.1442, 0.0000, 0.0000]])

In [12]:
(1 - X['mask'].to(torch.float32)) * X['meal']

tensor([[0.1229, 0.3462, 0.0000, 0.0000, 0.0000],
        [0.1583, 0.1133, 0.0000, 0.0000, 0.0000],
        [0.4140, 0.4759, 0.1120, 0.3140, 0.0000],
        [0.3768, 0.3454, 0.2862, 0.2951, 0.4730],
        [0.2358, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4005, 0.1133, 0.3252, 0.0000, 0.0000],
        [0.1357, 0.2049, 0.5176, 0.1069, 0.0000],
        [0.3100, 0.3902, 0.0000, 0.0000, 0.0000],
        [0.3454, 0.2568, 0.2951, 0.1232, 0.1069],
        [0.3836, 0.1229, 0.1357, 0.0000, 0.0000]])

tensor([[0.4852, 0.3577, 0.4634, 0.0000, 0.0000],
        [0.1120, 0.1229, 0.3055, 0.3462, 0.0000],
        [0.1120, 0.4634, 0.3100, 0.0000, 0.0000],
        [0.3454, 0.3100, 0.1279, 0.2568, 0.0000],
        [0.1188, 0.3100, 0.1069, 0.0000, 0.0000],
        [0.2680, 0.3100, 0.4332, 0.2812, 0.0000],
        [0.1232, 0.1000, 0.1229, 0.2446, 0.0000],
        [0.1232, 0.2686, 0.1583, 0.0000, 0.0000],
        [0.2686, 0.1232, 0.1080, 0.0000, 0.0000],
        [0.3462, 0.0000, 0.0000, 0.0000, 0.0000]])

In [6]:
class LitPOSForecast(L.LightningModule):
    def __init__(
        self,
        scaler,
        n_meal_types: int = 5,
        n_restaurants: int = 3,
        d_hid: int = 32,
        lr: float = 3e-4,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()

        self.scaler = scaler

        self.forecaster = POSForecast(
            n_meal_types=n_meal_types,
            n_restaurants=n_restaurants,
            d_hid=d_hid,
        )
        self.lr = lr

        self.mse = MeanSquaredError()
        self.r2 = R2Score()
        self.preds_val, self.tgts_val = [], []
        self.preds_train, self.tgts_train = [], []

    def training_step(self, batch, batch_idx):
        tgt_train = batch["tgt_train"]
        tgt = batch["tgt"]

        pred = self.forecaster(batch)

        loss = nn.functional.mse_loss(pred, tgt_train)
        self.log("train_loss", loss, prog_bar=True, on_step=True)

        self.preds_train.append(pred)
        self.tgts_train.append(tgt)

        return loss

    def on_train_epoch_end(self) -> None:
        preds = torch.concat(self.preds_train, dim=0).detach().cpu()
        preds = torch.tensor(
            self.scaler.inverse_transform(preds),
            dtype=torch.float32,
            device=self.device,
        )

        tgts = torch.concat(self.tgts_train, dim=0)

        rmse = torch.sqrt(self.mse(preds, tgts))
        r2 = self.r2(preds, tgts)

        self.log("rmse_train", rmse, on_epoch=True)
        self.log("r2_train", r2, on_epoch=True)

        self.preds_train, self.tgts_train = [], []

    def validation_step(self, batch, batch_idx):
        tgt = batch["tgt"]

        pred = self.forecaster(batch)

        self.preds_val.append(pred)
        self.tgts_val.append(tgt)

    def on_validation_epoch_end(self) -> None:
        preds = torch.tensor(
            self.scaler.inverse_transform(torch.concat(self.preds_val, dim=0).cpu()),
            dtype=torch.float32,
            device=self.device,
        )

        tgts = torch.concat(self.tgts_val, dim=0)

        rmse = torch.sqrt(self.mse(preds, tgts))
        r2 = self.r2(preds, tgts)

        self.log("rmse_val", rmse, on_epoch=True)
        self.log("r2_val", r2, on_epoch=True)

        # self.preds_val, self.tgts_val = [], []

    def configure_optimizers(self):
        optimizer = AdamW(self.parameters(), lr=self.lr)

        return optimizer


In [7]:
BATCH_SIZE = 256

loader_train = DataLoader(POSData(pos_train), batch_size=BATCH_SIZE, shuffle=True)
loader_val = DataLoader(POSData(pos_val), batch_size=BATCH_SIZE)

litmodel = LitPOSForecast.load_from_checkpoint("weights/idea5_TransEnc/11-29_14-46-08/epoch=05-rmse_val=61.73.ckpt")

trainer = L.Trainer()
trainer.validate(litmodel, loader_val)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/hoangle/Projects/fwo_models/.venv/lib/python3.11/site-packages/lightning/pytorch/trainer/connectors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.


Validation: |          | 0/? [00:00<?, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃      Validate metric      ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│          r2_val           │    0.13189034163951874    │
│         rmse_val          │     61.7330322265625      │
└───────────────────────────┴───────────────────────────┘

[{'rmse_val': 61.7330322265625, 'r2_val': 0.13189034163951874}]

In [10]:
preds = torch.tensor(
    litmodel.scaler.inverse_transform(torch.concat(litmodel.preds_val, dim=0).cpu()),
    dtype=torch.float32,
    device=litmodel.device,
)

tgts = torch.concat(litmodel.tgts_val, dim=0)

In [11]:
preds

tensor([[ 1.5561e+02,  1.5857e+02,  2.9554e+02, -5.3889e+01, -5.3889e+01],
        [ 1.5561e+02,  2.9640e+02,  1.5857e+02, -5.3889e+01, -5.3889e+01],
        [-9.5663e-02, -5.3889e+01, -5.3889e+01, -5.3889e+01, -5.3889e+01],
        ...,
        [ 1.5254e+02,  1.4849e+02,  3.0394e+02,  3.0750e+02,  1.5196e+02],
        [ 1.1783e+02,  2.0659e+02,  1.4870e+02,  1.1740e+02, -5.3889e+01],
        [ 1.2496e+02,  1.2509e+02,  1.9021e+02, -5.3889e+01, -5.3889e+01]])

In [12]:
tgts

tensor([[116.,  89., 453.,   0.,   0.],
        [116., 453.,  89.,   0.,   0.],
        [ 24.,   0.,   0.,   0.,   0.],
        ...,
        [ 40., 200.,  96., 325.,  85.],
        [177., 102., 114.,  27.,   0.],
        [ 44., 125., 106.,   0.,   0.]], device='mps:0')

In [ ]:
rmse = torch.sqrt(self.mse(preds, tgts))
        r2 = self.r2(preds, tgts)

### Start training

In [ ]:
BATCH_SIZE = 100
LR = 5e-4

loader_train = DataLoader(POSData(pos_train), batch_size=BATCH_SIZE, shuffle=True)
loader_val = DataLoader(POSData(pos_val), batch_size=BATCH_SIZE)

model = POSForecast()
litmodel = LitPOSForecast(model, LR, scaler)


trainer = L.Trainer(
    # devices=0,
    callbacks=[RichProgressBar(leave=True)],
    logger=TensorBoardLogger("tb_logs", name="idea5_TransEnc"),
    # gradient_clip_val=1,
    max_epochs=20,
)

trainer.fit(litmodel, loader_train, loader_val)

GPU available: True (mps), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


┏━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┓
┃   ┃ Name       ┃ Type             ┃ Params ┃ Mode  ┃
┡━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━┩
│ 0 │ forecaster │ POSForecast      │  3.1 M │ train │
│ 1 │ mse        │ MeanSquaredError │      0 │ train │
│ 2 │ r2         │ R2Score          │      0 │ train │
└───┴────────────┴──────────────────┴────────┴───────┘

Trainable params: 3.1 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 3.1 M                                                                                                
Total estimated model params size (MB): 12                                                                         
Modules in train mode: 55                                                                                          
Modules in eval mode: 0

Output()

/Users/hoangle/Projects/Food-Waste-Optimization/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connec
tors/data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider 
increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.

/var/folders/pr/8dv_cj95295bxt_hr8hzrmk40000gn/T/ipykernel_83607/1320065469.py:26: UserWarning: To copy construct 
from a tensor, it is recommended to use sourceTensor.clone().detach() or 
sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(meal == 0, dtype=torch.bool)

/Users/hoangle/Projects/Food-Waste-Optimization/.venv/lib/python3.10/site-packages/lightning/pytorch/trainer/connec
tors/data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider 
increasing the value of the `num_workers` argument` to `num_workers=9` in the `DataLoader` to improve performance.

/var/folders/pr/8dv_cj95295bxt_hr8hzrmk40000gn/T/ipykernel_83607/1320065469.py:26: UserWarning: To copy construct 
from a tensor, it is recommended to use sourceTensor.clone().detach() or 
sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(meal == 0, dtype=torch.bool)


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

In [ ]:
# train_ds = POSData(pos_train)
# # X = train_ds[0]
# loader_train = DataLoader(train_ds, batch_size=10, shuffle=True)
# for X in loader_train:
#     break

# meal = X['meal']
# meal_type = X['meal_type']
# mask = X['mask']
# restaurant = X['restaurant']
# date = X['date']
# sim = X['sim']

# # Encode
# meal_type = model._embd_meal_type(meal_type)
# restaurant = model._embd_restaurant(restaurant)

# date = model.lin_date(date)
# sim = model.lin_sim(sim.unsqueeze(-1))
# meal = model.lin_meal(meal.unsqueeze(-1))

# # Concate fields
# N = meal.shape[1]
# restaurant = torch.repeat_interleave(restaurant.unsqueeze(1), N, dim=1)
# date = torch.repeat_interleave(date.unsqueeze(1), N, dim=1)

# meals = torch.concat(
#     [
#         meal,
#         meal_type,
#         restaurant,
#         date,
#         sim,
#     ],
#     dim=-1
# )

# meals = model.ff_combine(meals)

# # Use Transformer Encoder
# mask = mask.unsqueeze(1).repeat_interleave(5, dim=1)
# meals = model.trans_encoder(meals, mask)

# # Predict pos
# pos = F.tanh(model.lin_pcs(meals).squeeze(-1))

# # pos = pos * mask[:, :, 0]

# pos

/var/folders/pr/8dv_cj95295bxt_hr8hzrmk40000gn/T/ipykernel_80245/1320065469.py:26: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  mask = torch.tensor(meal == 0, dtype=torch.bool)


tensor([[-0.5091, -0.4822, -0.6616, -0.4519, -0.4519],
        [ 0.2616,  0.3446,  0.3260,  0.2589,  0.2589],
        [-0.5241, -0.7562, -0.5732, -0.7337, -0.5199],
        [-0.4798, -0.5101, -0.7224, -0.5291, -0.4427],
        [-0.4916, -0.6287, -0.7698, -0.5465, -0.5465],
        [-0.5033, -0.5017, -0.5017, -0.5017, -0.5017],
        [ 0.1550,  0.2824,  0.2386,  0.2753,  0.2661],
        [-0.6949, -0.6772, -0.6381, -0.6381, -0.6381],
        [-0.7384, -0.6686, -0.4796, -0.4877, -0.4498],
        [-0.7292, -0.7298, -0.6616, -0.6637, -0.6637]],
       grad_fn=<TanhBackward0>)

In [ ]:
# embed_dim = 4
# mha = torch.nn.MultiheadAttention(embed_dim=embed_dim, num_heads=1, batch_first=True)

# # assume we have a batch of 2 sentences. 1st has 3 tokens and 2nd has 2 tokens
# embeddings = torch.normal(mean=0, std=1, size=(2, 3, embed_dim))
# # create a padding mask with all zeros so that every token is valid by default
# key_padding_mask = torch.zeros(size=(2, 3), dtype=torch.bool)
# # 3rd token of second sentence is a pad token
# key_padding_mask[1, 2] = 1

# _, torch_attn_mask = mha(embeddings, embeddings, embeddings, key_padding_mask=key_padding_mask)
# print(torch_attn_mask)

tensor([[[0.3869, 0.3286, 0.2845],
         [0.2554, 0.3867, 0.3579],
         [0.2925, 0.3620, 0.3455]],

        [[0.4998, 0.5002, 0.0000],
         [0.4115, 0.5885, 0.0000],
         [0.5706, 0.4294, 0.0000]]], grad_fn=<MeanBackward1>)


In [ ]:
# # reshape mask to proper shape
# key_padding_mask_expanded = key_padding_mask.unsqueeze(1) # (bs, 1, seq_len)
# # expand 3 times in the 2nd dimension since we have 3 tokens
# key_padding_mask_expanded = key_padding_mask_expanded.expand(-1, 3, -1)
# print(key_padding_mask_expanded)


tensor([[[False, False, False],
         [False, False, False],
         [False, False, False]],

        [[False, False,  True],
         [False, False,  True],
         [False, False,  True]]])


In [35]:
key_padding_mask

tensor([[False, False, False],
        [False, False,  True]])